<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z345_ExploratorioCategoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratorio por Categoría — impulso, decay y canibalización

Para productos de la misma categoría (cat3):
- Series de tiempo interactivas con Plotly
- Detección de productos nuevos (nacen dentro de los 36 meses)
- Fuerza del impulso inicial y velocidad de decay
- Mean reversion: ¿el producto vuelve a un nivel estable?
- Clustering por forma dentro de la misma categoría
- Canibalización: ¿cuando uno sube, otro baja?

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell
mkdir -p "/content/.drive/My Drive/labo3" /content/buckets
ln -sfn "/content/.drive/My Drive/labo3" /content/buckets/b1
mkdir -p /content/buckets/b1/datasets /content/datasets
descargar() {
  d="/content/buckets/b1/datasets/"
  u="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  if ! test -f "$d$1"; then wget "$u$1" -O "$d$1"; fi
  if ! test -f "/content/datasets/$1"; then cp "$d$1" "/content/datasets/$1"; fi
}
descargar sell-in.txt.gz
descargar tb_productos.txt
descargar product_id_apredecir201912.txt

In [ ]:
!pip install uv -q && uv pip install -q plotly kaleido

In [ ]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── PARAM ────────────────────────────────────────────────────────
PARAM = {
    # Nivel de categoría a usar: 'cat1' (más amplio) / 'cat2' / 'cat3' (más fino)
    'nivel_cat':        'cat3',

    # Categoría a explorar — None = mostrar ranking y elegir
    'categoria':        None,

    # Cuántos productos mostrar en el ranking inicial
    'top_n_ranking':    20,

    # Meses iniciales para medir el impulso de nacimiento
    'ventana_impulso':  6,

    # Meses recientes para medir el nivel actual
    'ventana_reciente': 3,

    # Período completo del dataset
    'periodo_ini':      201801,
    'periodo_fin':      201912,
}
print('OK')

# 1. Carga de datos

In [ ]:
dataset     = pl.read_csv('/content/datasets/sell-in.txt.gz', separator='\t')
tb_prod     = pl.read_csv('/content/datasets/tb_productos.txt', separator='\t')
tb_apredecir = pl.read_csv('/content/datasets/product_id_apredecir201912.txt', separator='\t')

# ventas agregadas por producto y periodo
tb_ventas = (
    dataset
    .group_by('product_id', 'periodo')
    .agg(pl.col('tn').sum())
    .sort(['product_id', 'periodo'])
)

# join con productos para tener categorías
tb_ventas = tb_ventas.join(tb_prod, on='product_id', how='left')

col_cat = PARAM['nivel_cat']
categorias_disponibles = tb_ventas[col_cat].drop_nulls().unique().sort().to_list()

print(f'Períodos: {tb_ventas["periodo"].min()} – {tb_ventas["periodo"].max()}')
print(f'Productos únicos: {tb_ventas["product_id"].n_unique()}')
print(f'Categorías ({col_cat}): {len(categorias_disponibles)}')
print()
print(tb_prod.head(5))

# 2. Ranking de categorías — elegí cuál explorar

In [ ]:
ranking_cat = (
    tb_ventas
    .group_by(col_cat)
    .agg(
        pl.col('tn').sum().alias('tn_total'),
        pl.col('product_id').n_unique().alias('n_productos'),
    )
    .sort('n_productos', descending=True)
)

print(f'Top {PARAM["top_n_ranking"]} categorías por cantidad de productos:')
print(ranking_cat.head(PARAM['top_n_ranking']))

# elegir categoría
if PARAM['categoria'] is None:
    # autoselección: la categoría con más productos
    CAT_ELEGIDA = ranking_cat[col_cat][0]
    print(f'\nAutoseleccionada: {CAT_ELEGIDA}')
else:
    CAT_ELEGIDA = PARAM['categoria']
    print(f'\nCategoría elegida: {CAT_ELEGIDA}')

# 3. Series de tiempo interactivas — Plotly

In [ ]:
# filtrar la categoría elegida
tb_cat = tb_ventas.filter(pl.col(col_cat) == CAT_ELEGIDA).sort(['product_id', 'periodo'])
productos_cat = tb_cat['product_id'].unique().sort().to_list()

print(f'Categoría: {CAT_ELEGIDA}  |  {len(productos_cat)} productos')

# grilla completa periodos × productos
todos_periodos = sorted(tb_ventas['periodo'].unique().to_list())

fig = go.Figure()

colores = px.colors.qualitative.Plotly
for i, pid in enumerate(productos_cat):
    serie_df = tb_cat.filter(pl.col('product_id') == pid).sort('periodo')
    periodos = serie_df['periodo'].to_list()
    tn_vals  = serie_df['tn'].to_list()
    desc     = serie_df['descripcion'][0] if 'descripcion' in serie_df.columns else str(pid)

    fig.add_trace(go.Scatter(
        x=[str(p) for p in periodos],
        y=tn_vals,
        mode='lines+markers',
        name=f'{pid} — {desc}',
        line=dict(color=colores[i % len(colores)], width=1.8),
        marker=dict(size=4),
        hovertemplate=f'<b>{pid}</b><br>periodo: %{{x}}<br>tn: %{{y:.2f}}<extra></extra>',
    ))

fig.update_layout(
    title=f'Series de tiempo — categoría: {CAT_ELEGIDA}',
    xaxis_title='Periodo',
    yaxis_title='tn',
    hovermode='x unified',
    height=500,
    legend=dict(font=dict(size=10)),
)
fig.show()

# 4. Detección de productos nuevos — nacen dentro del período

In [ ]:
PERIODO_INICIO_DATASET = todos_periodos[0]

info_productos = []
for pid in productos_cat:
    serie_df  = tb_cat.filter(pl.col('product_id') == pid).sort('periodo')
    periodos  = serie_df['periodo'].to_list()
    tn        = serie_df['tn'].to_numpy().astype(float)
    desc      = str(serie_df['descripcion'][0]) if 'descripcion' in serie_df.columns else str(pid)
    brand     = str(serie_df['brand'][0]) if 'brand' in serie_df.columns else ''
    sku_size  = str(serie_df['sku_size'][0]) if 'sku_size' in serie_df.columns else ''

    primer_periodo = periodos[0]
    es_nuevo = primer_periodo > PERIODO_INICIO_DATASET

    # meses desde inicio del dataset hasta primer periodo
    idx_nacimiento = todos_periodos.index(primer_periodo) if primer_periodo in todos_periodos else 0

    # impulso: max en los primeros ventana_impulso meses con dato
    v_imp = PARAM['ventana_impulso']
    impulso_max    = float(tn[:v_imp].max()) if len(tn) >= 1 else 0.0
    impulso_media  = float(tn[:v_imp].mean()) if len(tn) >= 1 else 0.0

    # nivel reciente
    v_rec = PARAM['ventana_reciente']
    nivel_reciente = float(tn[-v_rec:].mean()) if len(tn) >= v_rec else float(tn.mean())

    # decay: caída desde el pico
    pico = float(tn.max())
    mes_pico = int(np.argmax(tn))
    ratio_decay = nivel_reciente / (pico + 1e-9)
    es_spike_inicial = mes_pico < v_imp

    # pendiente OLS reciente vs histórica
    def pendiente_ols(y):
        if len(y) < 2: return 0.0
        x = np.arange(len(y)).reshape(-1,1)
        return float(LinearRegression().fit(x, y).coef_[0])

    pend_historica = pendiente_ols(tn)
    pend_reciente  = pendiente_ols(tn[-6:]) if len(tn) >= 6 else pend_historica

    info_productos.append({
        'product_id':      pid,
        'descripcion':     desc,
        'brand':           brand,
        'sku_size':        sku_size,
        'primer_periodo':  primer_periodo,
        'es_nuevo':        es_nuevo,
        'idx_nacimiento':  idx_nacimiento,
        'n_meses':         len(tn),
        'impulso_max':     impulso_max,
        'impulso_media':   impulso_media,
        'nivel_reciente':  nivel_reciente,
        'pico':            pico,
        'mes_pico':        mes_pico,
        'ratio_decay':     ratio_decay,
        'es_spike_inicial':es_spike_inicial,
        'pend_historica':  pend_historica,
        'pend_reciente':   pend_reciente,
        'serie':           tn,
        'periodos':        periodos,
    })

tb_info = pl.DataFrame([{k: v for k, v in r.items() if k not in ('serie', 'periodos')} for r in info_productos])
print(f'Productos nuevos (nacen después de {PERIODO_INICIO_DATASET}): {tb_info["es_nuevo"].sum()}')
print(f'Productos con spike inicial: {tb_info["es_spike_inicial"].sum()}')
print()
display(tb_info.select(['product_id','descripcion','primer_periodo','es_nuevo','impulso_max','ratio_decay','es_spike_inicial','pend_reciente']))

# 5. Fuerza del impulso y decay — Plotly interactivo

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Impulso inicial vs nivel reciente',
        'Ratio decay (nivel_reciente / pico)  — 1.0 = no decayó',
    ]
)

for r in info_productos:
    color = '#e74c3c' if r['es_spike_inicial'] else '#3498db'
    marker_symbol = 'star' if r['es_nuevo'] else 'circle'

    fig.add_trace(go.Scatter(
        x=[r['impulso_max']],
        y=[r['nivel_reciente']],
        mode='markers+text',
        marker=dict(size=10, color=color, symbol=marker_symbol,
                    line=dict(width=1, color='black')),
        text=[str(r['product_id'])],
        textposition='top center',
        textfont=dict(size=8),
        name=r['descripcion'],
        hovertemplate=(
            f"<b>{r['product_id']} — {r['descripcion']}</b><br>"
            f"impulso_max: {r['impulso_max']:.2f}<br>"
            f"nivel_reciente: {r['nivel_reciente']:.2f}<br>"
            f"ratio_decay: {r['ratio_decay']:.3f}<br>"
            f"spike_inicial: {r['es_spike_inicial']}<extra></extra>"
        ),
        showlegend=False,
    ), row=1, col=1)

# línea de referencia: impulso = reciente (sin decay)
max_imp = max(r['impulso_max'] for r in info_productos)
fig.add_trace(go.Scatter(
    x=[0, max_imp], y=[0, max_imp],
    mode='lines', line=dict(dash='dash', color='gray', width=1),
    name='sin decay', showlegend=True,
), row=1, col=1)

# panel derecho: ratio decay por producto
pids_sorted = sorted(info_productos, key=lambda r: r['ratio_decay'])
fig.add_trace(go.Bar(
    x=[str(r['product_id']) for r in pids_sorted],
    y=[r['ratio_decay'] for r in pids_sorted],
    marker_color=['#e74c3c' if r['ratio_decay'] < 0.3 else
                  '#f39c12' if r['ratio_decay'] < 0.7 else '#2ecc71'
                  for r in pids_sorted],
    hovertemplate='%{x}<br>ratio_decay=%{y:.3f}<extra></extra>',
    showlegend=False,
), row=1, col=2)

fig.add_hline(y=0.3, line_dash='dot', line_color='red',   annotation_text='decay fuerte', row=1, col=2)
fig.add_hline(y=0.7, line_dash='dot', line_color='orange', annotation_text='decay moderado', row=1, col=2)

fig.update_layout(
    title=f'Impulso y decay — {CAT_ELEGIDA}  |  ★=producto nuevo  ●rojo=spike inicial',
    height=500,
)
fig.update_xaxes(title_text='impulso_max (tn)', row=1, col=1)
fig.update_yaxes(title_text='nivel_reciente (tn)', row=1, col=1)
fig.show()

# 6. Series de productos con spike inicial — zoom en el decay

In [ ]:
spikes = [r for r in info_productos if r['es_spike_inicial']]

if not spikes:
    print('No hay productos con spike inicial en esta categoría.')
else:
    fig = go.Figure()
    for r in spikes:
        tn_norm = r['serie'] / (r['pico'] + 1e-9)  # normalizar por pico para comparar forma
        fig.add_trace(go.Scatter(
            x=list(range(len(tn_norm))),
            y=tn_norm.tolist(),
            mode='lines+markers',
            name=f"{r['product_id']} — {r['descripcion']}",
            marker=dict(size=4),
            hovertemplate=f"<b>{r['product_id']}</b><br>mes: %{{x}}<br>tn/pico: %{{y:.3f}}<extra></extra>",
        ))

    fig.add_hline(y=0.3, line_dash='dot', line_color='red',
                  annotation_text='30% del pico')
    fig.update_layout(
        title=f'Curvas de decay normalizadas por pico — {CAT_ELEGIDA}<br>'
              f'(eje Y = tn / pico_histórico, 1.0 = momento del pico)',
        xaxis_title='Mes desde primer dato',
        yaxis_title='tn / pico',
        height=450,
        hovermode='x unified',
    )
    fig.show()
    print(f'{len(spikes)} productos con spike inicial:')
    for r in sorted(spikes, key=lambda x: x['ratio_decay']):
        print(f"  {r['product_id']:6d}  {r['descripcion']:40s}  ratio_decay={r['ratio_decay']:.3f}  mes_pico={r['mes_pico']}")

# 7. Canibalización — correlación de Pearson entre productos

In [ ]:
# Alinear todas las series al mismo eje temporal (grilla completa)
# Productos sin dato en un período → 0

grilla = {}
for r in info_productos:
    vec = np.zeros(len(todos_periodos))
    for p, v in zip(r['periodos'], r['serie']):
        if p in todos_periodos:
            vec[todos_periodos.index(p)] = v
    grilla[r['product_id']] = vec

pids = list(grilla.keys())
n    = len(pids)

# matriz de Pearson
mat_pearson = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        v1, v2 = grilla[pids[i]], grilla[pids[j]]
        if v1.std() > 0 and v2.std() > 0:
            r_val, _ = pearsonr(v1, v2)
            mat_pearson[i, j] = r_val
        else:
            mat_pearson[i, j] = 0.0

labels_heatmap = [f"{r['product_id']}\n{r['descripcion'][:15]}" for r in info_productos]

fig = go.Figure(go.Heatmap(
    z=mat_pearson,
    x=[str(p) for p in pids],
    y=[str(p) for p in pids],
    colorscale='RdYlGn',
    zmin=-1, zmax=1,
    text=[[f'{mat_pearson[i,j]:.2f}' for j in range(n)] for i in range(n)],
    hovertemplate='%{y} vs %{x}<br>Pearson: %{z:.3f}<extra></extra>',
    colorbar=dict(title='Pearson'),
))
fig.update_layout(
    title=f'Correlación de Pearson entre productos — {CAT_ELEGIDA}<br>'
          f'Verde=se mueven juntos (complementarios) | Rojo=se mueven opuestos (canibalizan)',
    height=max(400, n*40),
    width=max(500, n*50),
)
fig.show()

# 8. Clustering por forma — ¿qué productos se comportan igual?

In [ ]:
if n < 2:
    print('Necesitás al menos 2 productos para clustering.')
else:
    # normalizar por max para comparar formas
    vectores_norm = []
    for pid in pids:
        v = grilla[pid]
        m = v.max()
        vectores_norm.append(v / m if m > 0 else v)

    Z = linkage(vectores_norm, method='ward')

    n_clusters = min(max(2, n // 3), 6)
    labels_cluster = fcluster(Z, n_clusters, criterion='maxclust')

    fig, ax = plt.subplots(figsize=(max(10, n*0.8), 6))
    descs = [next(r['descripcion'][:20] for r in info_productos if r['product_id'] == p) for p in pids]
    dendrogram(
        Z,
        labels=[f"{p}\n{d}" for p, d in zip(pids, descs)],
        ax=ax,
        leaf_rotation=45,
        leaf_font_size=8,
    )
    ax.set_title(f'Clustering por forma de serie (norm=max) — {CAT_ELEGIDA}', fontsize=11)
    ax.set_ylabel('distancia Ward')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'Asignación a {n_clusters} clusters:')
    for pid, cl in zip(pids, labels_cluster):
        desc = next(r['descripcion'] for r in info_productos if r['product_id'] == pid)
        print(f'  cluster {cl}  |  {pid:6d}  {desc}')

# 9. Series por cluster — visualización

In [ ]:
if n >= 2:
    fig = make_subplots(
        rows=1, cols=n_clusters,
        subplot_titles=[f'Cluster {c}' for c in range(1, n_clusters+1)],
    )
    colores = px.colors.qualitative.Plotly

    for cl in range(1, n_clusters+1):
        miembros = [pids[i] for i, lbl in enumerate(labels_cluster) if lbl == cl]
        for pid in miembros:
            r_info = next(r for r in info_productos if r['product_id'] == pid)
            v = grilla[pid]
            m = v.max()
            v_norm = v / m if m > 0 else v
            fig.add_trace(go.Scatter(
                x=[str(p) for p in todos_periodos],
                y=v_norm.tolist(),
                mode='lines',
                name=f"{pid} — {r_info['descripcion'][:20]}",
                line=dict(width=1.5),
                hovertemplate=f'<b>{pid}</b><br>%{{x}}: %{{y:.3f}}<extra></extra>',
                showlegend=(cl == 1),
            ), row=1, col=cl)

    fig.update_layout(
        title=f'Series normalizadas por cluster — {CAT_ELEGIDA}',
        height=400,
        hovermode='x unified',
    )
    fig.show()

# 10. Resumen — tabla de features por producto

In [ ]:
if n >= 2:
    clusters_dict = {pids[i]: int(labels_cluster[i]) for i in range(n)}
    tb_resumen = tb_info.with_columns(
        pl.col('product_id').map_elements(lambda p: clusters_dict.get(p, -1), return_dtype=pl.Int32).alias('cluster')
    ).sort('cluster')
else:
    tb_resumen = tb_info

display(tb_resumen.select([
    'product_id', 'descripcion', 'brand', 'sku_size',
    'primer_periodo', 'es_nuevo', 'n_meses',
    'impulso_max', 'nivel_reciente', 'ratio_decay',
    'es_spike_inicial', 'mes_pico', 'pend_historica', 'pend_reciente',
    'cluster',
]))

print()
print('Interpretación ratio_decay:')
print('  < 0.30 → decay fuerte    (el nivel actual es menos del 30% del pico)')
print('  0.30–0.70 → decay moderado')
print('  > 0.70 → estable          (el nivel actual es similar al pico)')

# Cambiar categoría

```python
PARAM['categoria']   = 'Mayonesa'   # nombre exacto de cat3
PARAM['nivel_cat']   = 'cat2'       # o cat1 para más productos
PARAM['ventana_impulso']  = 3       # meses para medir impulso inicial
PARAM['ventana_reciente'] = 6       # meses para medir nivel actual
```

Correr desde la celda **# 2** para recargar con la nueva categoría.